In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

Step 1 - Load data and price dataframe

In [2]:
xlu = pd.read_csv('/Users/Sinthu/Desktop/Quant Trading Project/data/xlu.csv', parse_dates=['Date'], date_format='%m/%d/%y', index_col='Date')   # change if needed
xlp = pd.read_csv('/Users/Sinthu/Desktop/Quant Trading Project/data/xlp.csv', parse_dates=['Date'], date_format='%m/%d/%y', index_col='Date')   # change if needed

xlu.columns = xlu.columns.str.strip()
xlp.columns = xlp.columns.str.strip()

xlu = pd.DataFrame({'Adj_Close': xlu['Close']})
xlp = pd.DataFrame({'Adj_Close': xlp['Close']})

xlu.sort_index(inplace=True)
xlp.sort_index(inplace=True)

In [3]:
prices = pd.DataFrame({
    "xlu": xlu["Adj_Close"],
    "xlp": xlp["Adj_Close"]
}).dropna()

In [4]:
prices["ratio"] = prices["xlu"] / prices["xlp"]

lookback = 60
prices["ratio_mean"] = prices["ratio"].rolling(lookback).mean()
prices["ratio_std"] = prices["ratio"].rolling(lookback).std()

prices["zscore"] = (prices["ratio"] - prices["ratio_mean"]) / prices["ratio_std"]

In [5]:
#Raw signal

entry_threshold = 2
exit_threshold = 0.5

prices["signal_raw"] = 0
prices.loc[prices["zscore"] < -entry_threshold, "signal_raw"] = 1
prices.loc[prices["zscore"] > entry_threshold, "signal_raw"] = -1

prices["signal_raw"].value_counts(dropna=False)

signal_raw
 0    6047
-1     460
 1     370
Name: count, dtype: int64

In [6]:
prices["position"] = 0

for i in range(1, len(prices)):
    prev_pos = prices["position"].iloc[i-1]
    z = prices["zscore"].iloc[i]
    raw = prices["signal_raw"].iloc[i]
    
    if prev_pos == 0:
        prices.iloc[i, prices.columns.get_loc("position")] = raw
    else:
        if abs(z) <= exit_threshold:
            prices.iloc[i, prices.columns.get_loc("position")] = 0
        else:
            prices.iloc[i, prices.columns.get_loc("position")] = prev_pos

In [7]:
#To avoid look ahead bias

prices["position_shifted"] = prices["position"].shift(1)

In [8]:
# Returns

prices["ret_xlu"] = prices["xlu"].pct_change()
prices["ret_xlp"] = prices["xlp"].pct_change()
prices = prices.dropna()

In [9]:
# Gross strategy returns

prices.loc[:,"spread_return_gross"] = prices["position_shifted"] * (prices["ret_xlu"] - prices["ret_xlp"])

In [10]:
transaction_cost = 0.001

prices["trade"] = prices["position"].diff().abs().fillna(0)
prices["cost"] = prices["trade"] * transaction_cost

prices["spread_return_net"] = prices["spread_return_gross"] - prices["cost"]

In [11]:
#Equity curves

prices["equity_gross"] = (1 + prices["spread_return_gross"]).cumprod()
prices["equity_net"] = (1 + prices["spread_return_net"]).cumprod()

In [12]:
#Flat baseline

prices["flat_return"] = 0.0
prices["flat_equity"] = (1 + prices["flat_return"]).cumprod()

Step 1 - Decide the parameters to test

## Sensitivity test setup

Choosing a small grid of values to test:
- Entry thresholds: 1.5, 2.0 and 2.5
- Exit thresholds: 0.0, 0.5 and 1.0
- Lookback windows: 30, 60 and 90

For our previous signal we used a 60 day lookback and +2/-2 entry threshold - this will now be out baseline.

Testing to see whether nearby values produce similar results or not.

Step 2 - Rebuilding the strategy inside a function

To test parameters properly, we need a reusuable function

In [13]:
def backtest_pair_strategy(prices, lookback=60, entry_threshold=2.0, exit_threshold=0.5, transaction_cost=0.001):
    temp = prices.copy()
    
    temp["ratio"] = temp["xlu"] / temp["xlp"]
    temp["ratio_mean"] = temp["ratio"].rolling(lookback).mean()
    temp["ratio_std"] = temp["ratio"].rolling(lookback).std()
    temp["zscore"] = (temp["ratio"] - temp["ratio_mean"]) / temp["ratio_std"]
    
    temp["signal_raw"] = 0
    temp.loc[temp["zscore"] < -entry_threshold, "signal_raw"] = 1
    temp.loc[temp["zscore"] > entry_threshold, "signal_raw"] = -1
    
    temp["position"] = 0
    for i in range(1, len(temp)):
        prev_pos = temp["position"].iloc[i-1]
        z = temp["zscore"].iloc[i]
        raw = temp["signal_raw"].iloc[i]
        if prev_pos == 0:
            temp.iloc[i, temp.columns.get_loc("position")] = raw
        else:
            if abs(z) <= exit_threshold:
                temp.iloc[i, temp.columns.get_loc("position")] = 0
            else:
                temp.iloc[i, temp.columns.get_loc("position")] = prev_pos
    
    temp["position_shifted"] = temp["position"].shift(1)
    temp["ret_xlu"] = temp["xlu"].pct_change()
    temp["ret_xlp"] = temp["xlp"].pct_change()
    temp["trade"] = temp["position"].diff().abs().fillna(0)
    temp["spread_return_gross"] = temp["position_shifted"] * (temp["ret_xlu"] - temp["ret_xlp"])
    temp["spread_return_net"] = temp["spread_return_gross"] - temp["trade"] * transaction_cost
    temp = temp.dropna()
    
    equity = (1 + temp["spread_return_net"]).cumprod()
    
    annual_return = (1 + temp["spread_return_net"].mean())**252 - 1
    annual_vol = temp["spread_return_net"].std() * np.sqrt(252)
    sharpe = np.nan if annual_vol == 0 else annual_return / annual_vol
    max_dd = (equity / equity.cummax() - 1).min()
    
    return {
        "lookback": lookback,
        "entry_threshold": entry_threshold,
        "exit_threshold": exit_threshold,
        "annual_return": annual_return,
        "annual_vol": annual_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "trade_count": int((temp["trade"] > 0).sum())
    }

WHY THIS MATTERS:
- This makes our strategy testable across different parameters/ settings instead of locking us in one setting.

Step 3 - Run the parameter grid

In [15]:
#Grid search

results = []

for lookback in [30, 60, 90]:
    for entry in [1.5, 2.0, 2.5]:
        for exit_th in [0.0, 0.5, 1.0]:
            if exit_th < entry:
                res = backtest_pair_strategy(prices, lookback=lookback, entry_threshold=entry, exit_threshold=exit_th)
                results.append(res)

results_df = pd.DataFrame(results)
results_df.head()

,lookback,entry_threshold,exit_threshold,annual_return,annual_vol,sharpe,max_drawdown,trade_count
0,30,1.5,0.0,-0.005445,0.158390,-0.034379,-0.565378,1
1,30,1.5,0.5,0.019979,0.115745,0.172612,-0.377635,590
2,30,1.5,1.0,0.018060,0.103162,0.175066,-0.376956,784
3,30,2.0,0.0,-0.003890,0.158117,-0.024600,-0.565378,1
4,30,2.0,0.5,0.022670,0.102405,0.221380,-0.334648,404


In [ ]:
results_df.head(27)

In [ ]:
print((results_df["annual_return"] > 0).sum())
print((results_df["annual_return"] <= 0).sum())

WHAT TO LOOK FOR:

Are many combinations profitable?
- 
Or only one small cluster?

Does performance collapse outside the baseline?